In [ ]:
import os
import logging
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- ibis_filter ---
_IBIS_DATA_DIR = next(
    candidate
    for root in [Path.cwd(), *Path.cwd().parents]
    for candidate in (
        root / "data/first_filter_data_local",
        root / "first_filter_data_local",
    )
    if candidate.exists()
)
_IBIS_FIXTURE_ROOT = _IBIS_DATA_DIR / "AroneyS__binchicken/test/data/mock_coassemble/coassemble"
_IBIS_CHECKM_SOURCE = _IBIS_FIXTURE_ROOT / "coassemble/coassembly_0/recover/bins/checkm_minimal.tsv"
_IBIS_CHECKM_FIXTURE = Path("/tmp/ibis_checkm_minimal_with_bin_id.tsv")
_ibis_checkm_df = pd.read_csv(_IBIS_CHECKM_SOURCE, sep="	").rename(columns={"Name": "Bin Id"})
_ibis_checkm_df.to_csv(_IBIS_CHECKM_FIXTURE, sep="	", index=False)
FIX_IBIS_FILTER_CHECKM_OUT_DICT = {"coassembly_0": str(_IBIS_CHECKM_FIXTURE)}
FIX_IBIS_FILTER_COASSEMBLY = "coassembly_0"
FIX_IBIS_FILTER_COMPLETENESS_COL = "Completeness"
FIX_IBIS_FILTER_CONTAMINATION_COL = "Contamination"
FIX_IBIS_FILTER_MAX_CONTAMINATION = 5.0
FIX_IBIS_FILTER_MIN_COMPLETENESS = 50.0

# --- ibis_isin ---
FIX_IBIS_ISIN_ARGS = SimpleNamespace(
    elusive_clusters=str(_IBIS_FIXTURE_ROOT / "target/elusive_clusters.tsv"),
    coassemblies=["coassembly_0"],
)

# --- ibis_join ---
FIX_IBIS_JOIN_CLUSTER = str(_IBIS_FIXTURE_ROOT / "target/elusive_clusters.tsv")
_IBIS_JOIN_NEW_CLUSTER_ROWS = {"coassembly": ["coassembly_0"], "samples": ["sample_1,sample_2"], "length": [2]}
FIX_IBIS_JOIN_NEW_CLUSTER_BEFORE = pd.DataFrame(_IBIS_JOIN_NEW_CLUSTER_ROWS)
FIX_IBIS_JOIN_NEW_CLUSTER_GEN = pl.DataFrame(_IBIS_JOIN_NEW_CLUSTER_ROWS)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_ibis_filter(checkm_out_dict, coassembly, completeness_col, contamination_col, max_contamination, min_completeness):
    checkm_out = pd.read_csv(checkm_out_dict[coassembly], sep = "\t")
    passed_bins = checkm_out[(checkm_out[completeness_col] >= min_completeness) & (checkm_out[contamination_col] <= max_contamination)]["Bin Id"].to_list()
    return passed_bins

def before_ibis_isin(args):
    elusive_clusters = pd.read_csv(os.path.abspath(args.elusive_clusters), sep="\t")
    elusive_clusters = elusive_clusters[elusive_clusters["coassembly"].isin(args.coassemblies)]
    return elusive_clusters

def before_ibis_join(cluster, new_cluster):
    old_cluster = pd.read_csv(cluster, sep="\t")
    comb_cluster = (
        new_cluster
        .set_index("samples")[["coassembly"]]
        .join(old_cluster.set_index("samples")[["length"]], how="inner")
        )
    if not comb_cluster.empty:
        comb_cluster.apply(lambda x: logging.warn(f"{x['coassembly']} has been previously suggested"), axis=1)
    return comb_cluster

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_ibis_filter(checkm_out_dict, coassembly, completeness_col, contamination_col, max_contamination, min_completeness):

    checkm_out = pl.read_csv(checkm_out_dict[coassembly], separator="\t")
    passed_bins = checkm_out.filter(
        (pl.col(completeness_col) >= min_completeness)
        & (pl.col(contamination_col) <= max_contamination)
    ).get_column("Bin Id").to_list()
    return passed_bins

def gen_ibis_isin(args):
    import os

    elusive_clusters = pl.read_csv(os.path.abspath(args.elusive_clusters), separator="\t")
    elusive_clusters = elusive_clusters.filter(pl.col("coassembly").is_in(args.coassemblies))
    return elusive_clusters

def gen_ibis_join(cluster, new_cluster):
    import logging

    old_cluster = pl.read_csv(cluster, separator="\t")
    comb_cluster = (
        new_cluster.select(["samples", "coassembly"])
        .join(old_cluster.select(["samples", "length"]), on="samples", how="inner")
    )
    if not comb_cluster.is_empty():
        for x in comb_cluster.iter_rows(named=True):
            logging.warn(f"{x['coassembly']} has been previously suggested")
    return comb_cluster

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: ibis_join ===

# L1 smoke – generated
try:
    _r = gen_ibis_join(FIX_IBIS_JOIN_CLUSTER, FIX_IBIS_JOIN_NEW_CLUSTER_GEN)
    print("✅ L1 smoke gen_ibis_join: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_ibis_join: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_ibis_join(FIX_IBIS_JOIN_CLUSTER, FIX_IBIS_JOIN_NEW_CLUSTER_BEFORE)
    print("✅ L1 smoke before_ibis_join: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_ibis_join: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_ibis_join(FIX_IBIS_JOIN_CLUSTER, FIX_IBIS_JOIN_NEW_CLUSTER_BEFORE)
    _rg = gen_ibis_join(FIX_IBIS_JOIN_CLUSTER, FIX_IBIS_JOIN_NEW_CLUSTER_GEN)
    compare(_rb, _rg, "ibis_join")
except Exception as _e:
    print(f"❌ L2 equivalence ibis_join: setup error — {type(_e).__name__}: {_e}")

# L3 edge - a schema-bearing empty new-cluster frame on both sides.
try:
    _empty_pd = pd.DataFrame({
        "coassembly": pd.Series(dtype="object"),
        "samples": pd.Series(dtype="object"),
        "length": pd.Series(dtype="int64"),
    })
    _empty_pl = pl.DataFrame(schema={
        "coassembly": pl.String, "samples": pl.String, "length": pl.Int64,
    })
    _rb = before_ibis_join(FIX_IBIS_JOIN_CLUSTER, _empty_pd)
    _rg = gen_ibis_join(FIX_IBIS_JOIN_CLUSTER, _empty_pl)
    compare(_rb, _rg, "L3 edge ibis_join empty new cluster", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge ibis_join empty new cluster: {type(_e).__name__}: {_e}")
